# 1. Preprocessing and exploratory data analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = Path("../data")
FIGURES_PATH = Path("../figures")
FIGURES_PATH.mkdir(exist_ok=True)

ModuleNotFoundError: No module named 'pandas'

## Load dataset

In [ ]:
bank_data = pd.read_csv(DATA_PATH / "bank-additional-full.csv", sep=";")
bank_data.shape

In [ ]:
bank_data.head()

## Data types and missing values

In [ ]:
bank_data.info()

## Proportion of "unknown" values per categorical column

In [ ]:
categorical_columns = bank_data.select_dtypes(exclude="number").columns
(bank_data[categorical_columns] == "unknown").mean().sort_values(ascending=False)

## Remove duplicate rows

In [ ]:
bank_data.duplicated().sum()

In [ ]:
bank_data = bank_data.drop_duplicates()
bank_data.shape

## Target encoding: yes → 1, no → 0

In [ ]:
bank_data["y"] = (bank_data["y"] == "yes").astype(int)
bank_data["y"].value_counts(normalize=True)

## 80/20 stratified split of data in train and test

In [ ]:
train_data, test_data = train_test_split(
    bank_data,
    test_size=0.2,
    stratify=bank_data["y"],
    random_state=RANDOM_STATE,
)
print(train_data.shape, test_data.shape)
print(train_data["y"].mean(), test_data["y"].mean())

## Save raw train and test sets

In [ ]:
train_data.to_csv(DATA_PATH / "train.csv", index=False)
test_data.to_csv(DATA_PATH / "test.csv", index=False)

# EDA (train set only)

## Class balance

In [ ]:
class_proportions = train_data["y"].value_counts(normalize=True)

ax = class_proportions.plot(kind="bar", color=["tab:blue", "tab:orange"])
ax.set_xticklabels(["no (0)", "yes (1)"], rotation=0)
ax.set_ylabel("Proportion")
ax.set_title("Class balance (train)")
for position, proportion in enumerate(class_proportions):
    ax.text(position, proportion + 0.01, f"{proportion:.1%}", ha="center")
plt.savefig(FIGURES_PATH / "01_class_balance.png", bbox_inches="tight")
plt.show()

## Numerical variables: distribution by class

In [ ]:
numerical_columns = train_data.select_dtypes(include="number").columns.drop("y")

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, column in zip(axes.flat, numerical_columns):
    sns.histplot(data=train_data, x=column, hue="y", stat="density",
                 common_norm=False, bins=30, ax=ax)
    ax.set_title(column)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "02_numerical_distributions_by_class.png", bbox_inches="tight")
plt.show()

## Mean of numerical variables: clients who accept vs. do not accept

In [ ]:
train_data.groupby("y")[numerical_columns].mean().T.round(2)

## Data leakage: call duration is only known after the call

In [ ]:
sns.boxplot(data=train_data, x="y", y="duration", showfliers=False)
plt.title("Call duration by class (train)")
plt.savefig(FIGURES_PATH / "03_duration_leakage.png", bbox_inches="tight")
plt.show()

## pdays: 999 means the client was never contacted before

In [ ]:
never_contacted = train_data["pdays"] == 999
print(f"Never contacted: {never_contacted.mean():.1%}")
train_data.groupby(never_contacted)["y"].mean()

## Categorical variables: subscription rate per category

In [ ]:
overall_subscription_rate = train_data["y"].mean()

fig, axes = plt.subplots(2, 5, figsize=(24, 9))
for ax, column in zip(axes.flat, categorical_columns.drop("y")):
    sns.barplot(data=train_data, x=column, y="y", errorbar=None, ax=ax)
    ax.axhline(overall_subscription_rate, color="red", linestyle="--")
    ax.set_ylabel("Subscription rate")
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "04_subscription_rate_by_category.png", bbox_inches="tight")
plt.show()

## Rare categories (less than 100 samples)

In [ ]:
for column in categorical_columns.drop("y"):
    category_counts = train_data[column].value_counts()
    rare_categories = category_counts[category_counts < 100]
    if len(rare_categories) > 0:
        print(column, rare_categories.to_dict())

## Correlation matrix of numerical variables

In [ ]:
correlation_matrix = train_data[numerical_columns.tolist() + ["y"]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix (train)")
plt.savefig(FIGURES_PATH / "05_correlation_matrix.png", bbox_inches="tight")
plt.show()

## Decisions from the EDA
- `duration`: removed (data leakage)
- `pdays`: replaced by binary `previously_contacted` (96% are 999)
- `emp.var.rate`, `euribor3m`: removed (correlation > 0.9 with `nr.employed`)
- `housing`, `loan`, `day_of_week`: removed (same subscription rate in every category)
- `"unknown"`: kept as its own category (e.g. `default` unknown → 5% vs 13%)
- Rare categories (< 100 samples): grouped in the pipeline (notebook 2)
- Numerical variables: standard scaling in the pipeline (notebook 2)
- Class imbalance (11% yes): stratified split and stratified k-fold

## Feature engineering (same row-wise transformation for train and test)

In [ ]:
COLUMNS_TO_DROP = ["duration", "pdays", "emp.var.rate", "euribor3m",
                   "housing", "loan", "day_of_week"]


def engineer_features(data):
    engineered_data = data.copy()
    engineered_data["previously_contacted"] = (engineered_data["pdays"] != 999).astype(int)
    return engineered_data.drop(columns=COLUMNS_TO_DROP)


train_processed = engineer_features(train_data)
test_processed = engineer_features(test_data)
print(train_processed.shape, test_processed.shape)
train_processed.columns.tolist()

## Save processed train and test sets

In [ ]:
train_processed.to_csv(DATA_PATH / "train_processed.csv", index=False)
test_processed.to_csv(DATA_PATH / "test_processed.csv", index=False)